#Final Project - Baseline Model


# 0. Setup

## 0.1. Libraries and Helper Functions

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification,AutoConfig, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0.2 Data Acquisition

### FakeNewsNet Dataset
FAKENEWSNET is an ongoing data collection project for fake news research. It consists of headlines and body texts of fake news articles based on BuzzFeed and PolitiFact. It also collects information about the social engagements of these articles from Twitter.

https://github.com/JinyanSu1/Fakenews-dataset

The orginial fake news and real news are used as HF and HR.
MF and MR are datasets created by adopting the SMP prompting technique via ChatGPT. Generated 97 and 4,084 LLM-generated fake news for PolitiFact and GossipCop, respectively.

The dataset was created based on politifact and gossipcop dataset from FakeNewsNet (https://github.com/KaiDMML/FakeNewsNet/tree/dependabot/pip/certifi-2022.12.7), where the news articles that don't contain title or description has already been excluded.

In [ ]:
# Function to read a JSON file
def load_json_file(file_path):
    """Load a JSON file"""
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
# Function to process FakeNewsNet data
def process_fakenewsnet_data(file_paths):
    """Read multiple JSON files and organize them into a DataFrame"""
    data_list = []
    label_map = {"HF": "human_fake", "HR": "human_real", "MF": "gpt_fake", "MR": "gpt_real"}

    for category, file_path in file_paths.items():
        data = load_json_file(file_path)
        source = "gossipcop" if "gossipcop" in category else "politifact"
        label = label_map.get(category.split("_")[-1], "unknown")

        for news_id, news_content in data.items():
            if 'text' in news_content:
                data_list.append({
                    "id": news_id,
                    "text": news_content["text"],
                    "title": news_content.get("title", ""),
                    "source": source,
                    "label": label,
                })

    return pd.DataFrame(data_list)


In [ ]:
# Define dataset paths
fake_news_paths = {category: f"/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/FakeNewsNet/{category}.json"
                    for category in ["gossipcop_HF", "gossipcop_HR", "politifact_HF", "politifact_HR",
                                     "gossipcop_MF", "gossipcop_MR", "politifact_MF", "politifact_MR"]}

# Load and process the data
FakeNewsNet_df = process_fakenewsnet_data(fake_news_paths)

# Display the first 5 rows
FakeNewsNet_df.head()

,id,text,title,source,label
0,0,✕ Close Meghan Markle and Prince Harry have an...,As it happened: Prince Harry and Meghan Markle...,gossipcop,human_fake
1,1,Kim Kardashian and Kanye West are pulling out ...,Kim & Kanye Install At-Home Panic Room After P...,gossipcop,human_fake
2,2,Prince Harry and Meghan currently live at Kens...,£1.4million spent renovating Prince Harry and ...,gossipcop,human_fake
3,3,They can't get enough of the Biebs on this sho...,Photos from Dancing With the Stars: Special Gu...,gossipcop,human_fake
4,4,Ben Affleck is keeping life with his three kid...,Jennifer Garner ‘Doesn’t Want’ Her Kids Around...,gossipcop,human_fake


 ## 0.3. Data Preparation
Perform essential preprocessing steps to prepare the fake news dataset for further analysis or modeling:
*   Missing Value Check
*   Text Cleaning, including:
  *   Converts all text to lowercase to maintain consistency
  *   Removes URLs (both http and www formats) that do not contribute to the text meaning
  *   Eliminates extra whitespace for cleaner formatting

In [ ]:
# check if there is any missing value
FakeNewsNet_df.isnull().sum()

,0
id,0
text,0
title,0
source,0
label,0


In [ ]:
# Function to clean text
def clean_text(text):
    """Clean text by removing URLs, extra spaces and converting to lowercase"""
    # Lowercase the text
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
# Apply the text cleaning function
FakeNewsNet_df["cleaned_text"] = FakeNewsNet_df["text"].apply(clean_text)
# Display the first 5 rows
FakeNewsNet_df.head()

,id,text,title,source,label,cleaned_text
0,0,✕ Close Meghan Markle and Prince Harry have an...,As it happened: Prince Harry and Meghan Markle...,gossipcop,human_fake,✕ close meghan markle and prince harry have an...
1,1,Kim Kardashian and Kanye West are pulling out ...,Kim & Kanye Install At-Home Panic Room After P...,gossipcop,human_fake,kim kardashian and kanye west are pulling out ...
2,2,Prince Harry and Meghan currently live at Kens...,£1.4million spent renovating Prince Harry and ...,gossipcop,human_fake,prince harry and meghan currently live at kens...
3,3,They can't get enough of the Biebs on this sho...,Photos from Dancing With the Stars: Special Gu...,gossipcop,human_fake,they can't get enough of the biebs on this sho...
4,4,Ben Affleck is keeping life with his three kid...,Jennifer Garner ‘Doesn’t Want’ Her Kids Around...,gossipcop,human_fake,ben affleck is keeping life with his three kid...


 ### 0.3.1 Balance the dataset
Balance the dataset by retaining only the cleaned_text and label columns. Then, sampling each label category to ensure equal representation (4181 samples each). This helps prevent model bias caused by class imbalance.

In [ ]:
# Filter the relevant columns
FakeNewsNet_df = FakeNewsNet_df[["cleaned_text", "label"]]

In [ ]:
# check each label category
FakeNewsNet_df['label'].value_counts()

,count
label,
human_real,8362
gpt_real,4301
human_fake,4181
gpt_fake,4181


In [ ]:
# balance the data
human_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_real'].sample(n=4181, random_state=42)
gpt_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_real'].sample(n=4181, random_state=42)
human_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_fake']
gpt_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_fake']

### 0.3.2 Split Data into Train, Validation, and Test Sets
Split each label category into training (80%), validation (10%), and test (10%) sets using stratified sampling to preserve label distribution. The resulting subsets are then concatenated and shuffled to form the final balanced train, validation, and test datasets.

In [ ]:
# Split each label dataset into train (80%), validation (10%), and test (10%)
def split_dataset(df, label_name):
    train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
    return train, val, test

# Apply the split function to each label category
human_real_train, human_real_val, human_real_test = split_dataset(human_real_df, "human_real")
human_fake_train, human_fake_val, human_fake_test = split_dataset(human_fake_df, "human_fake")
gpt_real_train, gpt_real_val, gpt_real_test = split_dataset(gpt_real_df, "gpt_real")
gpt_fake_train, gpt_fake_val, gpt_fake_test = split_dataset(gpt_fake_df, "gpt_fake")

# Stack the train, validation, and test sets
train_df = pd.concat([human_real_train, human_fake_train, gpt_fake_train, gpt_real_train]).sample(frac=1, random_state=42)
val_df = pd.concat([human_real_val, human_fake_val, gpt_fake_val, gpt_real_val]).sample(frac=1, random_state=42)
test_df = pd.concat([human_real_test, human_fake_test, gpt_fake_test, gpt_real_test]).sample(frac=1, random_state=42)

# Print label distributions
print("Training set label distribution:")
print(train_df['label'].value_counts())
print("\nValidation set label distribution:")
print(val_df['label'].value_counts())
print("\nTest set label distribution:")
print(test_df['label'].value_counts())

# Print dataset sizes
print(f"Training: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")



Training set label distribution:
label
gpt_fake      3344
human_real    3344
gpt_real      3344
human_fake    3344
Name: count, dtype: int64

Validation set label distribution:
label
human_fake    418
gpt_real      418
human_real    418
gpt_fake      418
Name: count, dtype: int64

Test set label distribution:
label
gpt_fake      419
gpt_real      419
human_real    419
human_fake    419
Name: count, dtype: int64
Training: 13376, Validation: 1672, Test: 1676


# 1. Baseline Model - Bert & Bert-variant Models
Define a complete pipeline to train and evaluate transformer-based models for fake news classification. It includes:



*   Model & Tokenizer Loading: A function to load pre-trained models (like RoBERTa, DeBERTa, ModernBERT) and their tokenizers with custom output label size
*   Tokenization & Padding: A helper to tokenize and pad input texts, handling model-specific requirements (e.g., token type IDs for BERT/DeBERTa)
*   Dataset Class: A custom PyTorch Dataset (FakeNewsDataset) for feeding tokenized inputs and labels into the model.
*   Training & Evaluation: The train_and_evaluate function trains the model using Hugging Face’s Trainer, evaluates it on the test set. The evaluation metrics used includes:
  *   Accuracy, Precision, Recall, F1-score
  *   Classification report
  *   Confusion matrix
  *   Misclassification rate
*   Model Comparison: The pipeline is used to evaluate 6 models: ModernBERT-base, ModernBERT-large, roberta-base, roberta-large, deberta-v3-base, deberta-v3-large

## Model Setup

In [ ]:
# Tokenizer and model function
def load_model_and_tokenizer(model_name, num_labels):
    """Load tokenizer and model for a given model name."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = num_labels

    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    return tokenizer, model, config.model_type


# Tokenization function
def tokenize_and_pad(texts, tokenizer, model_type="bert", max_length=512):
    """Tokenize and pad the text data, ensuring compatibility with different model architectures."""
    return_token_type_ids = model_type in ["bert", "deberta"]

    encoding = tokenizer(
        texts,
        truncation=True,  # Ensure texts longer than max_length are truncated
        padding="max_length",  # Pad texts to max_length
        max_length=max_length,
        return_tensors="pt",
        return_token_type_ids=return_token_type_ids  # Only return token_type_ids for BERT/DeBERTa
    )
    return encoding

In [ ]:
# PyTorch Dataset class
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels, use_fp16):
        self.encodings = encodings
        self.labels = labels
        self.use_fp16 = use_fp16

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}

        if self.use_fp16:
            item["attention_mask"] = item["attention_mask"].to(dtype=torch.float16)

        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
def train_and_evaluate(model_name, num_labels, train_df, val_df, test_df, use_fp16=False):
    """Train and evaluate the model on fake news classification."""

    # Load tokenizer and model, and get model type
    tokenizer, model, model_type = load_model_and_tokenizer(model_name, num_labels)

    # Tokenization (pass model_type)
    train_encodings = tokenize_and_pad(train_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)
    val_encodings = tokenize_and_pad(val_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)
    test_encodings = tokenize_and_pad(test_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)

    # Label Mapping
    label_map = {"human_real": 0, "human_fake": 1, "gpt_real": 2, "gpt_fake": 3}
    train_labels = train_df['label'].map(label_map).tolist()
    val_labels = val_df['label'].map(label_map).tolist()
    test_labels = test_df['label'].map(label_map).tolist()

    # Create Dataset
    train_dataset = FakeNewsDataset(train_encodings, train_labels, use_fp16)
    val_dataset = FakeNewsDataset(val_encodings, val_labels, use_fp16)
    test_dataset = FakeNewsDataset(test_encodings, test_labels, use_fp16)

    # Training Arguments
    training_args = TrainingArguments(
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
        fp16=use_fp16,
        label_smoothing_factor=0.1
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Train
    trainer.train()

    # Save Model
    model_path = f"/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/{model_name}_base_model"
    trainer.save_model(model_path)

    # Evaluate
    eval_results = trainer.evaluate(test_dataset)
    print(f"Evaluation Results for {model_name}: {eval_results}")

    # Predict
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)

    # Classification Report
    report = classification_report(test_labels, y_pred, target_names=list(label_map.keys()))
    print(f"Classification Report for {model_name}:\n{report}")

    # Confusion Matrix
    conf_matrix = confusion_matrix(test_labels, y_pred)
    print(f"Confusion Matrix for {model_name}:\n{conf_matrix}")

    # Misclassification Rate
    misclassification_rate = 1 - accuracy_score(test_labels, y_pred)
    print(f"Misclassification Rate for {model_name}: {misclassification_rate:.4f}")


In [ ]:
# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Model Training

In [ ]:
# Train and evaluate RoBERTa, BERT, DeBERTa, and ModernBERT model
train_and_evaluate("answerdotai/ModernBERT-base", 4, train_df, val_df, test_df, use_fp16=True)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.778900,0.636174,0.841507,0.857692,0.841507,0.841635
2,0.576400,0.610309,0.856459,0.864011,0.856459,0.858500


<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Evaluation Results for answerdotai/ModernBERT-base: {'eval_loss': 0.6150106191635132, 'eval_accuracy': 0.8514319809069213, 'eval_precision': 0.8580525346258759, 'eval_recall': 0.8514319809069213, 'eval_f1': 0.8524737019048307, 'eval_runtime': 20.7219, 'eval_samples_per_second': 80.88, 'eval_steps_per_second': 5.067, 'epoch': 2.0}


<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Classification Report for answerdotai/ModernBERT-base:
              precision    recall  f1-score   support

  human_real       0.71      0.83      0.76       419
  human_fake       0.82      0.72      0.77       419
    gpt_real       0.92      0.87      0.90       419
    gpt_fake       0.98      0.98      0.98       419

    accuracy                           0.85      1676
   macro avg       0.86      0.85      0.85      1676
weighted avg       0.86      0.85      0.85      1676

Confusion Matrix for answerdotai/ModernBERT-base:
[[349  56  14   0]
 [106 302   9   2]
 [ 39  10 365   5]
 [  0   1   7 411]]
Misclassification Rate for answerdotai/ModernBERT-base: 0.1486


In [ ]:
# Train and evaluate RoBERTa, BERT, DeBERTa, and ModernBERT model
train_and_evaluate("answerdotai/ModernBERT-large", 4, train_df, val_df, test_df, use_fp16=True)

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.729800,0.606618,0.855263,0.865788,0.855263,0.855063
2,0.524900,0.575800,0.870215,0.874775,0.870215,0.871649


<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6

Evaluation Results for answerdotai/ModernBERT-large: {'eval_loss': 0.5883880257606506, 'eval_accuracy': 0.8663484486873508, 'eval_precision': 0.8727253617732375, 'eval_recall': 0.8663484486873508, 'eval_f1': 0.8679628907129024, 'eval_runtime': 6.7115, 'eval_samples_per_second': 249.722, 'eval_steps_per_second': 15.645, 'epoch': 2.0}


<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Classification Report for answerdotai/ModernBERT-large:
              precision    recall  f1-score   support

  human_real       0.72      0.83      0.77       419
  human_fake       0.84      0.77      0.80       419
    gpt_real       0.94      0.88      0.91       419
    gpt_fake       0.99      0.98      0.99       419

    accuracy                           0.87      1676
   macro avg       0.87      0.87      0.87      1676
weighted avg       0.87      0.87      0.87      1676

Confusion Matrix for answerdotai/ModernBERT-large:
[[349  55  14   1]
 [ 92 323   4   0]
 [ 42   7 368   2]
 [  0   1   6 412]]
Misclassification Rate for answerdotai/ModernBERT-large: 0.1337


In [ ]:
train_and_evaluate("roberta-base", 4, train_df, val_df, test_df, use_fp16=True)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (2 outputs left out)

<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Evaluation Results for roberta-base: {'eval_loss': 0.679210901260376, 'eval_accuracy': 0.8257756563245824, 'eval_precision': 0.8240013351194992, 'eval_recall': 0.8257756563245824, 'eval_f1': 0.8230354901983862, 'eval_runtime': 12.1289, 'eval_samples_per_second': 138.183, 'eval_steps_per_second': 8.657, 'epoch': 2.0}


<ipython-input-14-de781ed8b410>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Classification Report for roberta-base:
              precision    recall  f1-score   support

  human_real       0.71      0.74      0.73       419
  human_fake       0.80      0.68      0.73       419
    gpt_real       0.88      0.89      0.88       419
    gpt_fake       0.91      0.99      0.95       419

    accuracy                           0.83      1676
   macro avg       0.82      0.83      0.82      1676
weighted avg       0.82      0.83      0.82      1676

Confusion Matrix for roberta-base:
[[312  61  33  13]
 [104 284  16  15]
 [ 22  10 372  15]
 [  0   0   3 416]]
Misclassification Rate for roberta-base: 0.1742


In [ ]:
train_and_evaluate("roberta-large", 4, train_df, val_df, test_df, use_fp16=True)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


... (3 outputs left out)

<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Evaluation Results for roberta-large: {'eval_loss': 0.6165918707847595, 'eval_accuracy': 0.8615751789976134, 'eval_precision': 0.8675411397390388, 'eval_recall': 0.8615751789976134, 'eval_f1': 0.8618099152892361, 'eval_runtime': 5.1517, 'eval_samples_per_second': 325.327, 'eval_steps_per_second': 20.381, 'epoch': 2.0}


<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Classification Report for roberta-large:
              precision    recall  f1-score   support

  human_real       0.73      0.84      0.78       419
  human_fake       0.85      0.74      0.79       419
    gpt_real       0.96      0.87      0.91       419
    gpt_fake       0.94      1.00      0.97       419

    accuracy                           0.86      1676
   macro avg       0.87      0.86      0.86      1676
weighted avg       0.87      0.86      0.86      1676

Confusion Matrix for roberta-large:
[[354  50  12   3]
 [ 95 309   3  12]
 [ 38   6 364  11]
 [  0   0   2 417]]
Misclassification Rate for roberta-large: 0.1384


In [ ]:
train_and_evaluate("microsoft/deberta-v3-base", 4, train_df, val_df, test_df, use_fp16=False);

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (3 outputs left out)

<ipython-input-38-a7ce78afb8ea>:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  if key == "attention_mask" else torch.tensor(val[idx], dtype=torch.long)
<ipython-input-38-a7ce78afb8ea>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.float16 if self.use_fp16 else torch.float32)
<ipython-input-38-a7ce78afb8ea>:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  if key == "attention_mask" else torch.tensor(val[idx], dtype=torch.long)
<ipython-input-38-a7ce78afb8ea>:9: UserWarning: To copy cons

Evaluation Results for microsoft/deberta-v3-base: {'eval_loss': 0.645457923412323, 'eval_accuracy': 0.8394988066825776, 'eval_precision': 0.8382939606510119, 'eval_recall': 0.8394988066825776, 'eval_f1': 0.8380084143336807, 'eval_runtime': 32.916, 'eval_samples_per_second': 50.917, 'eval_steps_per_second': 3.19, 'epoch': 2.0}


<ipython-input-38-a7ce78afb8ea>:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  if key == "attention_mask" else torch.tensor(val[idx], dtype=torch.long)
<ipython-input-38-a7ce78afb8ea>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.float16 if self.use_fp16 else torch.float32)


Classification Report for microsoft/deberta-v3-base:
              precision    recall  f1-score   support

  human_real       0.73      0.76      0.74       419
  human_fake       0.82      0.74      0.77       419
    gpt_real       0.87      0.87      0.87       419
    gpt_fake       0.93      1.00      0.96       419

    accuracy                           0.84      1676
   macro avg       0.84      0.84      0.84      1676
weighted avg       0.84      0.84      0.84      1676

Confusion Matrix for microsoft/deberta-v3-base:
[[317  61  38   3]
 [ 89 308  14   8]
 [ 27   7 365  20]
 [  0   0   2 417]]
Misclassification Rate for microsoft/deberta-v3-base: 0.1605


In [ ]:
train_and_evaluate("microsoft/deberta-v3-large", 4, train_df, val_df, test_df, use_fp16=False);

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

... (4 outputs left out)

<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Evaluation Results for microsoft/deberta-v3-large: {'eval_loss': 0.5901145339012146, 'eval_accuracy': 0.866945107398568, 'eval_precision': 0.8689470899205963, 'eval_recall': 0.866945107398568, 'eval_f1': 0.8673463258800175, 'eval_runtime': 16.6459, 'eval_samples_per_second': 100.685, 'eval_steps_per_second': 6.308, 'epoch': 2.0}


<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Classification Report for microsoft/deberta-v3-large:
              precision    recall  f1-score   support

  human_real       0.74      0.79      0.77       419
  human_fake       0.81      0.78      0.79       419
    gpt_real       0.96      0.90      0.93       419
    gpt_fake       0.96      1.00      0.98       419

    accuracy                           0.87      1676
   macro avg       0.87      0.87      0.87      1676
weighted avg       0.87      0.87      0.87      1676

Confusion Matrix for microsoft/deberta-v3-large:
[[333  71  15   0]
 [ 85 326   1   7]
 [ 29   5 376   9]
 [  0   0   1 418]]
Misclassification Rate for microsoft/deberta-v3-large: 0.1331


In [ ]:
train_and_evaluate("bert-base-uncased", num_labels=4, train_df=train_df, val_df=val_df, test_df=test_df)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (2 outputs left out)

<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Evaluation Results for bert-base-uncased: {'eval_loss': 0.6823173761367798, 'eval_accuracy': 0.8192124105011933, 'eval_precision': 0.8162985332304045, 'eval_recall': 0.8192124105011933, 'eval_f1': 0.8156265320355344, 'eval_runtime': 24.704, 'eval_samples_per_second': 67.843, 'eval_steps_per_second': 4.25, 'epoch': 2.0}


<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Classification Report for bert-base-uncased:
              precision    recall  f1-score   support

  human_real       0.73      0.70      0.71       419
  human_fake       0.82      0.70      0.76       419
    gpt_real       0.82      0.88      0.85       419
    gpt_fake       0.89      0.99      0.94       419

    accuracy                           0.82      1676
   macro avg       0.82      0.82      0.82      1676
weighted avg       0.82      0.82      0.82      1676

Confusion Matrix for bert-base-uncased:
[[293  59  57  10]
 [ 83 295  18  23]
 [ 26   5 370  18]
 [  0   0   4 415]]
Misclassification Rate for bert-base-uncased: 0.1808


In [ ]:
train_and_evaluate("bert-large-uncased", num_labels=4, train_df=train_df, val_df=val_df, test_df=test_df)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

... (2 outputs left out)

<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}
<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Evaluation Results for bert-large-uncased: {'eval_loss': 0.6602748036384583, 'eval_accuracy': 0.837708830548926, 'eval_precision': 0.8353381832716533, 'eval_recall': 0.837708830548926, 'eval_f1': 0.8362153111414898, 'eval_runtime': 78.5652, 'eval_samples_per_second': 21.333, 'eval_steps_per_second': 1.336, 'epoch': 2.0}


<ipython-input-14-6a81041e66d8>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}


Classification Report for bert-large-uncased:
              precision    recall  f1-score   support

  human_real       0.73      0.71      0.72       419
  human_fake       0.79      0.78      0.78       419
    gpt_real       0.89      0.88      0.88       419
    gpt_fake       0.93      0.99      0.96       419

    accuracy                           0.84      1676
   macro avg       0.84      0.84      0.84      1676
weighted avg       0.84      0.84      0.84      1676

Confusion Matrix for bert-large-uncased:
[[296  80  35   8]
 [ 81 325   6   7]
 [ 29   6 368  16]
 [  0   0   4 415]]
Misclassification Rate for bert-large-uncased: 0.1623
